# Set up

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
#dbutils.widgets.text("catalog", "nikks_fevm_workspace_7405607030687545")
#dbutils.widgets.text("schema", "dataextraction")
#dbutils.widgets.text("table", "app")
#dbutils.widgets.text("volume", "/Volumes/main/nikkthegreek_iot/sharepoint")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table = dbutils.widgets.get("table")
volume = dbutils.widgets.get("volume")
volume = f"{volume}/invoices"

In [0]:
print(f"catalog: {catalog}, schema: {schema}, table: {table}, volume: {volume}")

# 1 Parse document

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{table}_parsed_exploration
COMMENT 'Table containing parsed exploration data from PDF files, including file metadata'
AS
    SELECT
        _metadata.file_path AS path,
        _metadata.file_name AS file_name,
        _metadata.file_size AS file_size,
        ai_parse_document(content, map(
        'version', '2.0',
        'descriptionElementTypes', '*'
        -- ,'imageOutputPath', '<UC Volume path>'
      )) AS parsed,
        parsed:document::string AS document -- only required when using ai_query
    FROM
        READ_FILES('{volume}', format => 'binaryFile', fileNamePattern => '*.pdf')
""")

In [0]:
display(spark.table(f"{catalog}.{schema}.{table}_parsed_exploration"))

# 2 AI query Extraction

In [0]:
prompt = """Extract ONLY the following three fields from this invoice and return them as a JSON object with no additional text or explanation:

1. invoice_date: The date of the invoice (format: YYYY-MM-DD)
2. invoice_sum: The total amount to be paid (numeric value only, no currency symbol)
3. seller: The name of the company issuing the invoice

Return ONLY valid JSON in this exact format:
{
  "invoice_date": "YYYY-MM-DD",
  "invoice_sum": 0.00,
  "seller": "Company Name"
}

Do not include any markdown, explanations, tables, or additional information."""

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{table}_ai_query_exploration AS
SELECT
    path,
    file_name,
    file_size,
    ai_query(
        endpoint => 'databricks-gpt-oss-120b',
        request => CONCAT('{prompt}', '\n\n', document),
        responseFormat => 'STRUCT<result: STRUCT<invoice_date: STRING, invoice_sum: DOUBLE, seller: STRING>>'
    ) AS ai_result
FROM {catalog}.{schema}.{table}_parsed_exploration
""")

In [0]:
display(spark.table(f"{catalog}.{schema}.{table}_ai_query_exploration"))

# 3 AI query Postprocessing

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{table}_ai_query_processed_exploration
COMMENT "The table contains processed data from invoices extracted from PDF documents. It includes details such as the file name of the PDF, the date of the invoice, the total invoice amount, and the seller's name. This table can be used for financial analysis, tracking invoice payments, and monitoring expenses related to different sellers."
AS
SELECT
    file_name,
    CAST(ai_result_struct.invoice_date AS DATE) AS invoice_date,
    ai_result_struct.invoice_sum,
    ai_result_struct.seller
FROM (
    SELECT
        file_name,
        from_json(ai_result, 'invoice_date DATE, invoice_sum DOUBLE, seller STRING') AS ai_result_struct
    FROM {catalog}.{schema}.{table}_ai_query_exploration
)
""")

In [0]:
spark.sql(f"ALTER TABLE {catalog}.{schema}.{table}_ai_query_processed_exploration ALTER COLUMN file_name COMMENT 'The name of the PDF file from which the invoice was extracted'")
spark.sql(f"ALTER TABLE {catalog}.{schema}.{table}_ai_query_processed_exploration ALTER COLUMN invoice_date COMMENT 'The date of the invoice (format: YYYY-MM-DD)'")
spark.sql(f"ALTER TABLE {catalog}.{schema}.{table}_ai_query_processed_exploration ALTER COLUMN invoice_sum COMMENT 'The total amount in Euros to be paid on the invoice (numeric value only)'")
spark.sql(f"ALTER TABLE {catalog}.{schema}.{table}_ai_query_processed_exploration ALTER COLUMN seller COMMENT 'The name of the company issuing the invoice'")

In [0]:
display(spark.table(f"{catalog}.{schema}.{table}_ai_query_processed_exploration"))

# 4 AI extract


In [0]:
complex_schema = """{
      "invoice_date": {"type": "string", "description": "Date of invoice in form YYYY-MM-DD"},
      "invoice_sum": {"type": "number", "description": "The total value to pay"},
      "seller": {"type": "string", "description": "Who was selling sth and created the invoice"}
    }"""

simple_schema = """["invoice_date", "invoice_sum", "seller"]"""

prompt = """I want to analyze private invoices"""

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{table}_ai_extract_exploration AS
SELECT
    path,
    file_name,
    file_size,
    ai_extract(
        parsed,
        schema => '{complex_schema}',
        options => map(
          'version', '2.0',
          'instructions', '{prompt}'
        )
    ) AS ai_result
FROM {catalog}.{schema}.{table}_parsed_exploration
""")

In [0]:
display(spark.table(f"{catalog}.{schema}.{table}_ai_extract_exploration"))

# 5 AI extract postprocessing


In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{table}_ai_extract_processed_exploration
COMMENT "The table contains processed data from invoices extracted from PDF documents. It includes details such as the file name of the PDF, the date of the invoice, the total invoice amount, and the seller's name. This table can be used for financial analysis, tracking invoice payments, and monitoring expenses related to different sellers."
AS
SELECT file_name, 
       ai_result:response.invoice_date::DATE AS invoice_date,
       ai_result:response.invoice_sum::DOUBLE AS invoice_sum,
       ai_result:response.seller::STRING AS seller
FROM main.nikkthegreek_iot.app_ai_extract_exploration
""")

In [0]:
display(spark.table(f"{catalog}.{schema}.{table}_ai_extract_processed_exploration"))